This analysis explores customer purchasing behavior using order history data. The objective was to understand how frequently customers place orders, shopping patterns, purchase timing, customer retention and lifespan to better understand customer engagement.

In [1]:
# Create a view that summarizes the number of orders per customer
spark.sql("""
CREATE OR REPLACE VIEW customers AS
SELECT
    user_id,
    COUNT(order_id) AS orders
FROM orders
GROUP BY user_id
""")

StatementMeta(, cb949050-442a-4d81-959d-c91c7ca343f7, 3, Finished, Available, Finished, False)

DataFrame[]

**Calculates total number of customers and the total number of orders**

In [11]:
Customers_count = spark.sql("""
SELECT COUNT(*) AS total_customers, 
SUM(orders) AS total_orders
FROM customers
""").toPandas()

Customers_count.style.format({'total_customers': '{:,}',
'total_orders': '{:,}'}).hide(axis='index')

StatementMeta(, cb949050-442a-4d81-959d-c91c7ca343f7, 13, Finished, Available, Finished, False)

total_customers,total_orders
"206,209","3,421,083"


_**206,209** unique customers, These customers collectively placed more than
**3,421,083** orders_

**How are customers distributed based on the number of orders they have placed?**

In [12]:
customer_order_ASC = spark.sql("""
SELECT
    orders, COUNT(user_id) AS customer_count,
    ROUND(COUNT(user_id) * 100.0/SUM(COUNT(user_id)) OVER (), 2) AS percentage
FROM customers
GROUP BY orders
ORDER BY orders ASC
LIMIT 5
""").toPandas()

customer_order_DESC = spark.sql("""
SELECT
    orders, COUNT(user_id) AS customer_count,
    ROUND(COUNT(user_id) * 100.0/ SUM(COUNT(user_id)) OVER (), 2) AS percentage
FROM customers
GROUP BY orders
ORDER BY orders DESC
LIMIT 5
""").toPandas()

display(customer_order_ASC.style.format({'customer_count': '{:,}', 
'percentage': '{:.2f}%'}).hide(axis='index'))

display(customer_order_DESC.style.format({'customer_count': '{:,}', 
'percentage': '{:.2f}%'}).hide(axis='index'))

StatementMeta(, cb949050-442a-4d81-959d-c91c7ca343f7, 14, Finished, Available, Finished, False)

orders,customer_count,percentage
4,"23,986",11.63%
5,"19,590",9.50%
6,"16,165",7.84%
7,"13,850",6.72%
8,"11,700",5.67%


orders,customer_count,percentage
100,"1,374",0.67%
99,47,0.02%
98,50,0.02%
97,54,0.03%
96,67,0.03%


**Customers were grouped into order frequency buckets based number of orders**

In [13]:
bucket_distribution = spark.sql("""
 SELECT
    bucket, COUNT(*) AS customers, 
    ROUND(COUNT(*) * 100.0 /SUM(COUNT(*)) OVER (), 2) AS percentage
FROM( SELECT
CASE
    WHEN orders BETWEEN 81 AND 100 THEN '81-100'
        WHEN orders BETWEEN 61 AND 80 THEN '61-80'
        WHEN orders BETWEEN 41 AND 60 THEN '41-60'
        WHEN orders BETWEEN 21 AND 40 THEN '21-40'
        ELSE '0-20'
    END AS bucket
FROM customers
) t
GROUP BY bucket
ORDER BY bucket
""").toPandas()

bucket_distribution.style.format({'customers': '{:,}',
'percentage': '{:.2f}%'}).hide(axis='index')

StatementMeta(, cb949050-442a-4d81-959d-c91c7ca343f7, 15, Finished, Available, Finished, False)

bucket,customers,percentage
0-20,"155,478",75.40%
21-40,"32,829",15.92%
41-60,"11,313",5.49%
61-80,"3,760",1.82%
81-100,"2,829",1.37%


_Most customers place relatively few orders. Customer count steadily decreases as lifetime order count increases.\
_A small group of highly engaged customers contributes a disproportionately large number of orders.__

**When do customers shop by days of week**

In [14]:
order_day_distribution = spark.sql("""
SELECT
    bucket, COUNT(*) AS orders,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM (SELECT
CASE
    WHEN order_day_of_week = 0 THEN 'Sunday'
    WHEN order_day_of_week = 1 THEN 'Monday'
    WHEN order_day_of_week = 2 THEN 'Tuesday'
    WHEN order_day_of_week = 3 THEN 'Wednesday'
    WHEN order_day_of_week = 4 THEN 'Thursday'
    WHEN order_day_of_week = 5 THEN 'Friday'
    ELSE 'Saturday'
END AS bucket
FROM orders
) t
GROUP BY bucket
ORDER BY bucket ASC
""").toPandas()

order_day_distribution.style.format({'orders': '{:,}', 
'percentage': '{:.2f}%'}) \
.hide(axis='index')

StatementMeta(, cb949050-442a-4d81-959d-c91c7ca343f7, 16, Finished, Available, Finished, False)

bucket,orders,percentage
Friday,"453,368",13.25%
Monday,"587,478",17.17%
Saturday,"448,761",13.12%
Sunday,"600,905",17.56%
Thursday,"426,339",12.46%
Tuesday,"467,260",13.66%
Wednesday,"436,972",12.77%


**When do customers shop by hours of day**

In [15]:
daypart_distribution = spark.sql("""
SELECT
    bucket, COUNT(*) AS orders,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM (SELECT
CASE
    WHEN order_hour BETWEEN 6 AND 11 THEN 'Morning'
    WHEN order_hour BETWEEN 12 AND 17 THEN 'Afternoon'
    WHEN order_hour BETWEEN 18 AND 21 THEN 'Evening'
    ELSE 'Night'
END AS bucket
FROM orders
) t
GROUP BY bucket
ORDER BY bucket DESC
""").toPandas()

daypart_distribution.style.format({'orders': '{:,}',
'percentage': '{:.2f}%'}).hide(axis='index')

StatementMeta(, cb949050-442a-4d81-959d-c91c7ca343f7, 17, Finished, Available, Finished, False)

bucket,orders,percentage
Night,"164,776",4.82%
Morning,"1,131,556",33.08%
Evening,"505,882",14.79%
Afternoon,"1,618,869",47.32%


**How often do customers return, what is the average gap between orders?** \
segment customers into:
Weekly (<7 days), Fortnightly (7–14), Monthly (14–30) and Occasional (30+ days)

In [17]:
shopping_frequency = spark.sql("""
SELECT
CASE
    WHEN avg_gap < 7 THEN 'Weekly (<7 days)'
    WHEN avg_gap < 14 THEN 'Fortnightly (7–14 days)'
    WHEN avg_gap < 30 THEN 'Monthly (14–30 days)'
    ELSE 'Occasional (30+ days)'
END AS shopping_frequency,
COUNT(*) AS customers,
ROUND(COUNT(*) * 100.0 /SUM(COUNT(*)) OVER (), 1) AS pct,
ROUND(AVG(avg_gap), 1) AS avg_days_in_segment
FROM( SELECT
    user_id,
    AVG(days_since_prior_order) AS avg_gap
FROM orders
WHERE days_since_prior_order IS NOT NULL
GROUP BY user_id
) t
GROUP BY shopping_frequency
ORDER BY avg_days_in_segment
""").toPandas()

shopping_frequency.style.format({'customers': '{:,}', 'pct': '{:.1f}%',
'avg_days_in_segment': '{:.1f}'}).hide(axis='index')    

StatementMeta(, cb949050-442a-4d81-959d-c91c7ca343f7, 19, Finished, Available, Finished, False)

shopping_frequency,customers,pct,avg_days_in_segment
Weekly (<7 days),"23,348",11.3%,5.2
Fortnightly (7–14 days),"69,391",33.7%,10.5
Monthly (14–30 days),"107,897",52.3%,20.1
Occasional (30+ days),"5,573",2.7%,30.0


_Customers were segmented by their average gap between orders into four frequency. The majority shop on a monthly (52.3%), while weekly shoppers the most engaged segment account for only 11.3% of the base_

**How long does a customer remain active from their first order?** \
Customer lifespan = cumulative days from first to last order

In [18]:
customer_lifespan = spark.sql("""
WITH lifespan AS (
SELECT
    user_id,
    SUM(COALESCE(days_since_prior_order, 0)) AS lifespan_days
FROM orders
GROUP BY user_id
)
SELECT
CASE
    WHEN lifespan_days < 30  THEN '<30 days'
    WHEN lifespan_days < 90  THEN '30–90 days'
    WHEN lifespan_days < 180 THEN '90–180 days'
    WHEN lifespan_days < 365 THEN '180–365 days'
    ELSE '365+ days'
END AS lifespan_bucket,
COUNT(*) AS customers,
ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct,
ROUND(AVG(lifespan_days), 0) AS avg_days_in_bucket
FROM lifespan
GROUP BY lifespan_bucket
ORDER BY avg_days_in_bucket
""").toPandas()

customer_lifespan.style.format({'customers': '{:,}', 'pct': '{:.1f}%',
'avg_days_in_bucket': '{:,.0f}'}).hide(axis='index')

StatementMeta(, cb949050-442a-4d81-959d-c91c7ca343f7, 20, Finished, Available, Finished, False)

lifespan_bucket,customers,pct,avg_days_in_bucket
<30 days,"4,844",2.3%,21
30–90 days,"47,388",23.0%,64
90–180 days,"67,744",32.9%,129
180–365 days,"84,665",41.1%,275
365+ days,"1,568",0.8%,365


_Customer lifespan was calculated as the cumulative sum of days_since_prior_order from first to last order. The 180–365 day bucket is the largest (41.1%, 84,665 customers), followed by 90–180 days (32.9%). Fewer than 1% of customers remain active beyond one year_